# Notebook 01 · 第一次跑通 VLM 推理

> 对应计划 **Day 1**。目标不是「跑通一个模型」，而是**看见一件反直觉的事**：
>
> **图片进入语言模型之前，会变成一堆 token —— 而且是一大堆。**
>
> 这件事决定了后面 8 周所有的工程取舍：为什么显存不够、为什么训练慢、
> 为什么「调小图片」比「调小 batch」有效。

---

### 怎么跑

**Colab**（不需要本地环境）：
1. 打开 https://colab.research.google.com
2. 上传这个 notebook，或者直接复制 cell 内容
3. 运行时 → 更改运行时类型 → 选 **T4 GPU**
4. 按顺序跑，**不要点「全部运行」** —— 中间有几个地方需要你看输出

**本地 Mac**：
```bash
pip install -r requirements-core.txt jupyterlab
jupyter lab notebooks/01_first_vlm_inference.ipynb
```
CPU 也能跑 3B（慢，一次推理 1-2 分钟），跑通流程够用。

**云 GPU 实例**：
```bash
bash scripts/cloud_bootstrap.sh
```

---

### 跑完你应该能回答

1. 一张 1024×1024 的图，最终变成多少个 token？
2. 这些 token 是**怎么**算出来的（哪两步）？
3. 一张图大概相当于多少个汉字？
4. 为什么说「视觉 token 太贵」是 VLM 的核心矛盾？


## 0 · 环境检查

两件事：**有没有 GPU**、**transformers 版本够不够**。

Qwen2.5-VL 需要 `transformers >= 4.49.0`。低于这个版本连 processor 都构建不了，
会报一些看不懂的错。所以先确认版本。

In [ ]:
import sys, platform

print("Python     :", sys.version.split()[0])
print("平台       :", platform.system(), platform.machine())

try:
    import torch
    print("torch      :", torch.__version__)
    print("CUDA 可用  :", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU        :", torch.cuda.get_device_name(0))
        print("显存       :", f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
        cap = torch.cuda.get_device_capability(0)
        print("bf16 支持  :", cap[0] >= 8)
    else:
        print("→ 没有 GPU，会跑在 CPU 上。慢，但流程一样。")
except ImportError:
    print("!! torch 没装。先跑下一格")

try:
    import transformers
    print("\ntransformers:", transformers.__version__)
    from packaging.version import Version
    if Version(transformers.__version__) < Version("4.49.0"):
        print("   !! 版本太低，Qwen2.5-VL 需要 >= 4.49.0")
        print("      修: %pip install -U 'transformers>=4.49.0'")
    else:
        print("   版本 OK")
except ImportError:
    print("!! transformers 没装")

In [ ]:
# 需要的话装一下（Colab 上跑这一格；本地已装就跳过）
%pip install -q "transformers>=4.49.0" "qwen-vl-utils>=0.0.10" accelerate pillow

# 国内网络建议配镜像，否则下载模型会极慢
import os
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
print("HF_ENDPOINT =", os.environ["HF_ENDPOINT"])

## 1 · 造一张「内容已知」的测试图

这一步很关键，**不要用随机噪声或者随便从网上拉一张图**。

为什么：如果你的测试图本身没有明确内容，模型答错了你分不清是
「模型不行」还是「图本来就没东西可看」。用一张你自己画、内容和颜色都确定的图，
你就能判断模型是真的看见了，还是在编。

下面画的是：**左边一个橙色圆，右边一个蓝色方块，底部一行字**。

In [ ]:
from PIL import Image, ImageDraw
import os

W = H = 448
img = Image.new("RGB", (W, H), (245, 245, 245))
d = ImageDraw.Draw(img)

d.ellipse([60, 110, 190, 240], fill=(230, 140, 60))      # 橙色圆
d.rectangle([250, 140, 385, 290], fill=(60, 110, 200))   # 蓝色方块
d.text((62, 330), "MMLAB SMOKE TEST", fill=(20, 20, 20)) # 一行字

os.makedirs("assets/samples", exist_ok=True)
path = "assets/samples/smoke_448.png"
img.save(path)

print(f"已保存: {path}  ({W}x{H}, mode={img.mode})")
img.resize((224, 224))  # 在 notebook 里显示一下

## 2 · 加载模型

我们用 **Qwen2.5-VL-3B-Instruct**：
- 3B 是能在单张 24 GB 卡上 LoRA 微调的甜点尺寸
- 中文和 OCR 都强（电商客服场景很需要看小字）
- 后面 8 周的后训练全部基于它

第一次跑要下载约 7 GB，走镜像一般 3-10 分钟。

In [ ]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)

# T4/V100 不支持 bf16，必须退回 fp16
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if use_bf16 else (torch.float16 if torch.cuda.is_available() else torch.float32)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"参数量 : {n_params/1e9:.2f} B")
print(f"dtype  : {dtype}")
print(f"设备   : {next(model.parameters()).device}")

## 3 · 看看模型的三个部分是哪些

讲义里说的「VLM = 视觉编码器 + 连接器 + 语言主干」，现在把它**在真实模型里指出来**。

这一段不用懂每一层，只要建立「哪些参数属于哪一部分」的直觉 ——
因为 Day 13 定 LoRA 的 target 时，你要决定**动哪一部分**。

In [ ]:
from collections import defaultdict

buckets = defaultdict(lambda: [0, 0])   # 前缀 -> [参数量, 层数]

for name, p in model.named_parameters():
    # 取前两个层级作为归类依据
    parts = name.split(".")
    key = ".".join(parts[:2])
    if name.startswith("visual"):
        key = "visual.*  (视觉编码器)"
    elif "merger" in name:
        key = "visual.merger  (连接器)"
    elif name.startswith("model.language_model") or name.startswith("model.") and "layers" in name:
        key = "lm.layers  (语言主干 layers)"
    elif name.startswith("lm_head"):
        key = "lm_head  (输出头)"
    buckets[key][0] += p.numel()
    buckets[key][1] += 1

total = sum(v[0] for v in buckets.values())
print(f"{'模块':38s} {'参数量':>12s} {'占比':>8s} {'张量数':>7s}")
print("-" * 70)
for k, (n, c) in sorted(buckets.items(), key=lambda x: -x[1][0])[:12]:
    print(f"{k:38s} {n/1e9:>10.3f} B {n/total*100:>7.1f}% {c:>7d}")
print("-" * 70)
print(f"{'合计':38s} {total/1e9:>10.3f} B")

## 4 · 关键的一步：图片变成多少 token？

现在到了这个 notebook 的核心。

我们要做的是：**把图片送进 processor，然后数一数 input_ids 里有多少个图像占位符**。

Qwen2.5-VL 用 `<|image_pad|>` 这个特殊 token 表示「这里要插入一个视觉特征」。
processor 会把图片处理成 N 个 patch，然后把对话里的一个 `<|image_pad|>` 展开成 N 个。

**这个 N，就是这张图的「token 价格」。**

In [ ]:
messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": path},
        {"type": "text", "text": "这张图里有哪些形状？分别是什么颜色？用一句话回答。"},
    ],
}]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("=== 模板渲染后的文本（注意 <|vision_start|> 那一段）===")
print(text[:600])
print("...")

In [ ]:
inputs = processor(text=[text], images=[img], return_tensors="pt")
inputs = inputs.to(model.device)

ids = inputs["input_ids"][0]
print("input_ids 形状 :", tuple(inputs["input_ids"].shape))
print("pixel_values   :", tuple(inputs["pixel_values"].shape), "(展平后的 patch 序列)")
print("image_grid_thw :", inputs["image_grid_thw"].tolist(), "(t,h,w 的网格尺寸)")

# 数一数视觉 token
img_token_id = model.config.image_token_id
n_visual = int((ids == img_token_id).sum())
n_text = len(ids) - n_visual

print()
print("=" * 52)
print(f"  总 token        : {len(ids)}")
print(f"  其中视觉 token  : {n_visual}   <-- 这张图的价格")
print(f"  其中文本 token  : {n_text}")
print(f"  视觉占比        : {n_visual/len(ids)*100:.1f}%")
print("=" * 52)

## 5 · 这个数字是怎么来的？

两种算法，**都应该得到同一个数**。这是 Day 4 的验收测试。

**算法一：从 grid 反推**
```
image_grid_thw = [t, h, w]        # h, w 是 patch 网格数
                                    # 每个 grid 单元 = patch_size (14) 像素
合并后的 token 数 = t * ceil(h/2) * ceil(w/2)
```
那个 `/2` 就是**「2×2 patch merge」** —— Qwen2.5-VL 在进 LLM 之前，
把相邻 2×2 共 4 个 patch 特征拼起来，用一个 MLP 压成 1 个 token。
这一步让 token 数直接少了 4 倍。

**算法二：自己算一遍**
```
1. smart_resize：把图片缩放到「能被 patch_size*merge 整除」的尺寸
2. patches = (H/14) * (W/14)
3. tokens  = patches / 4
```

下面两个都算一遍，看是否和上面数出来的一致。

In [ ]:
import math

t, gh, gw = inputs["image_grid_thw"][0].tolist()

method1 = t * math.ceil(gh / 2) * math.ceil(gw / 2)
print(f"算法一（从 grid 反推）: t={t}, h={gh}, w={gw}")
print(f"  {t} * ceil({gh}/2) * ceil({gw}/2) = {t} * {math.ceil(gh/2)} * {math.ceil(gw/2)} = {method1}")

PATCH, MERGE = 14, 2
aligned = PATCH * MERGE   # 28：因为最终尺寸要能被 28 整除
rh = math.ceil(H / aligned) * aligned
rw = math.ceil(W / aligned) * aligned
patches = (rh // PATCH) * (rw // PATCH)
method2 = patches // (MERGE ** 2)

print(f"\n算法二（自己算）:")
print(f"  原图 {H}x{W} -> 对齐后 {rh}x{rw}  (要能被 patch*merge={aligned} 整除)")
print(f"  patch 数 = ({rh}/14) * ({rw}/14) = {patches}")
print(f"  合并后   = {patches} / 4 = {method2}")

print()
print("-" * 52)
print(f"  processor 数出来的 : {n_visual}")
print(f"  算法一算出来的     : {method1}")
print(f"  算法二算出来的     : {method2}")
match = (n_visual == method1 == method2)
print("-" * 52)
if match:
    print("  ✅ 三个数一致 —— 你真的搞懂了 Qwen2.5-VL 的分辨率处理")
else:
    print("  ❌ 对不上。检查一下：图片是不是被 smart_resize 缩放过？")
    print("     原图尺寸 " + str((H, W)) + " 和 grid " + str((gh * PATCH, gw * PATCH)))

## 6 · 跑一次真实推理

前面都在算账，现在让模型真的看图。

注意看两件事：
1. **答对了吗**？（橙色圆 + 蓝色方块）
2. **视觉 token 占了多少**？如果超过 60%，说明这张图在「主导」整个输入 ——
   这就是服务成本的主要来源。

In [ ]:
import time

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

t0 = time.time()
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
dt = time.time() - t0

answer = processor.batch_decode(
    out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True
)[0].strip()

print("模型回答:")
print("  " + answer)
print()
print(f"耗时        : {dt:.1f} 秒")
print(f"新生成 token: {out.shape[1] - inputs['input_ids'].shape[1]}")
if dt > 0:
    print(f"生成速度    : {(out.shape[1] - inputs['input_ids'].shape[1])/dt:.1f} tok/s")
if torch.cuda.is_available():
    peak = torch.cuda.max_memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"峰值显存    : {peak:.2f} GB / {total:.1f} GB")

## 7 · 核心实验：token 数随图片尺寸怎么涨？

现在做这个 notebook 最有价值的一个实验。

**预测一下**：图片边长翻倍，token 数变成几倍？

想好了再跑下面这格。

In [ ]:
sizes = [224, 336, 448, 672, 896, 1024, 1280, 1536]

print(f"{'尺寸':>10s} {'像素数':>10s} {'patch数':>9s} {'视觉token':>10s} {'约等于汉字':>10s}")
print("-" * 60)

rows = []
for s in sizes:
    test = Image.new("RGB", (s, s), (200, 200, 200))
    inp = processor(text=[text], images=[test], return_tensors="pt")
    n = int((inp["input_ids"][0] == model.config.image_token_id).sum())
    t_, gh, gw = inp["image_grid_thw"][0].tolist()
    patches = t_ * gh * gw
    rows.append((s, n))
    # 一个中文 token 大约 1 个字（Qwen 的词表对中文比较友好），英文约 0.7 词
    print(f"{s}x{s:<7d} {s*s:>10d} {patches:>9d} {n:>10d} {n:>9d}字")

print()
if len(rows) >= 2:
    s1, n1 = rows[0]
    s2, n2 = rows[-1]
    ratio_dim = s2 / s1
    ratio_tok = n2 / n1
    print(f"边长 {s1} -> {s2}（{ratio_dim:.1f} 倍），"
          f"token {n1} -> {n2}（{ratio_tok:.1f} 倍）")
    print(f"→ token 数 ≈ 随边长的 {math.log(ratio_tok)/math.log(ratio_dim):.1f} 次方增长")
    print("   （略低于平方，因为 2x2 merge 的对齐取整在大尺寸下浪费更少）")

### 这段输出意味着什么

一张 **1024×1024** 的图 ≈ **1369 个 token**，大约等于 **1300 个汉字**。

什么概念：
- 一段 1300 字的客服对话，大概 5–8 轮
- 而你的模型上下文是 4096
- **所以 3 张图就吃掉一整个上下文窗口**

这就解释了后面 8 周会遇到的所有显存和成本问题：

| 现象 | 原因 |
|---|---|
| 训练时 OOM | 图片 token 顶爆序列长度，激活值随序列长度增长 |
| 训练很慢 | 每个样本要过视觉塔 + 更长的序列 |
| 服务成本高 | KV cache 大小和序列长度成正比 |
| **调小图片比调小 batch 有效** | 直接砍 token 数，而不是减少并发 |

所以 `IMAGEFACTOR` / `max_pixels` 这个看起来不起眼的参数，
实际上是整个系统里**性价比最高的旋钮**。`docs/04-qwen25vl.md` 有更细的分析。

## 8 · 练习（Day 1 的产出）

### 练习 1：验证 merge 的作用

把下面的 `MERGE` 改成 1（假设不做合并），token 数会变成几倍？
然后再想想：**如果真不做 merge，训练成本会变成多少倍？**

### 练习 2：OCR 能力探测（和客服场景强相关）

画一张带小字的图（模拟商品标签/尺码表），看模型能不能读出来。
然后**把字缩小一半再试**，找出它读不出来的临界点。

这个临界点很重要 —— 它决定了你在 W2 合成数据时，
图片要渲染成多大才不会一开始就超出模型能力。

### 练习 3：制造一次幻觉

问一个图里**没有**的东西：

```python
"这张图里有几只猫？"
```

注意看它是说「图里没有猫」，还是开始编。
**把这个回答存下来** —— 这是你这条路上的第一个 bad case，
W4 的幻觉评测和 W5 的 DPO 都会用到这类样本。

In [ ]:
# 练习 1：改 MERGE 看 token 变化
def tokens_if_merge(side, patch=14, merge=2):
    aligned = patch * merge
    r = math.ceil(side / aligned) * aligned
    return (r // patch) ** 2 // (merge ** 2)

for m in (1, 2, 4):
    n = tokens_if_merge(1024, merge=m)
    print(f"merge={m}: 1024x1024 -> {n:>5d} token")

print()
print("merge=2 相对 merge=1 省了",
      f"{tokens_if_merge(1024, merge=1)/tokens_if_merge(1024, merge=2):.1f} 倍 token")
print("→ 这就是为什么所有现代 VLM 都要做 token 压缩")

In [ ]:
# 练习 3：制造一次幻觉（把结果记下来，后面要用）
hallucination_probe = [{
    "role": "user",
    "content": [
        {"type": "image", "image": path},
        {"type": "text", "text": "这张图里有几只猫？它们在哪？"},
    ],
}]

t2 = processor.apply_chat_template(hallucination_probe, tokenize=False, add_generation_prompt=True)
inp2 = processor(text=[t2], images=[img], return_tensors="pt").to(model.device)

with torch.no_grad():
    o2 = model.generate(**inp2, max_new_tokens=80, do_sample=False)

ans2 = processor.batch_decode(o2[:, inp2["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()
print("问: 这张图里有几只猫？")
print("答:", ans2)
print()
print("─" * 56)
print("判断标准：")
print("  · 说「图里没有猫」 → 模型有基本的拒答意识")
print("  · 编出「两只白猫」 → 典型幻觉，这就是 W5 DPO 要治的病")
print()
print("把这个回答抄进 progress/daily-log.md 的 Day 1 那一栏。")

## 9 · Day 1 收尾

### 你现在应该能回答

1. **一张 1024×1024 的图变成多少 token？** ~1369 个
2. **怎么算出来的？** smart_resize 对齐 → 除以 14 得 patch 数 → 2×2 merge 除以 4
3. **相当于多少汉字？** ~1300 字
4. **为什么说视觉 token 太贵？** 3 张图就吃掉整个 4096 上下文

### 白板测试（合上电脑做）

画出这条路径，标注每一段的形状：

```
图片(1024x1024x3)
    ↓  
[ ? ]  ⇒ 输出形状 ?
    ↓
[ ? ]  ⇒ 输出形状 ?   ← 这一步把 4 个 patch 合成 1 个 token
    ↓
替换掉文本里的 <|?|> 占位符
    ↓
[ ? ]  ⇒ 输出 "图里有一个橙色圆形和一个蓝色方块"
```

画不出来就回去看 `docs/01-architecture.md` 的那张图。

### 打卡

在 `progress/daily-log.md` 里填 Day 1 那一栏，至少写清：
- 模型对「有几只猫」的回答是什么（第一个 bad case）
- 客服场景的 5 个图文问题

### 下一步

**Day 2**：`docs/02-vision-encoder.md` + 手写 `src/minivlm/vision.py`

明天要回答的新问题：
> 那个把图片变成向量的「视觉编码器」，内部到底在做什么？
> 为什么它用对比学习训练，而不是直接做分类？
